In [2]:
#1.PIIMiddleware
from langchain.agents.middleware import SummarizationMiddleware, PIIMiddleware
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

import dotenv
load_dotenv()
model=init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
)


In [3]:
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

agent=create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("email",strategy="redact",apply_to_input=True),
        PIIMiddleware("credit_card",strategy="mask",apply_to_input=True),
        PIIMiddleware("url",strategy="hash",apply_to_input=True),
        PIIMiddleware("mac_address",strategy="mask",apply_to_input=True),
        PIIMiddleware("ip",strategy="block",apply_to_input=True),
    ]
)
response=agent.invoke({
    "messages":[
        HumanMessage(""" 帮我向 156168188@qq.com 发送一封邮件
    同时查看银行卡号： 5105-1051-0510-5100 的余额
    访问 https://localhost:12345
    确认这是不是 MAC地址： 11-11-11-11-11-11""")
    ]
})
for msg in  response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

 帮我向 [REDACTED_EMAIL] 发送一封邮件
    同时查看银行卡号： ****-****-****-5100 的余额
    访问 <url_hash:dd5fc2a9>
    确认这是不是 MAC地址： **-**-**-**-**-11
================================== Ai Message ==================================

抱歉，我不能帮你完成这些涉及隐私和账户安全的操作：

- **发送邮件**：收件人已被打码，且没有具体内容和你的授权说明。
- **查询银行卡余额**：我不能访问银行账户，也强烈不建议把完整卡号提供给任何人/AI。
- **访问链接**：`<url_hash:dd5fc2a9>` 不是可直接访问的完整网址，我也不会代为访问未知链接。
- **确认 MAC 地址**：`**-**-**-**-**-11` 中 `**` 不是合法的十六进制字符，所以它不是有效的 MAC 地址格式。  
  有效格式示例：`00-1A-2B-3C-4D-5E`。

如果有其他非敏感问题，我可以帮你解答。


In [4]:
try:
    response1=agent.invoke({
        "messages":[HumanMessage("看看这个IP能不能ping通过：192.168.10.1")]
    })
except Exception as e:
    print(f"检测到ip,抛出异常{e}")

检测到ip,抛出异常Detected 1 instance(s) of ip in text content


In [ ]:
#举例2 自定义函数检测器/函数